In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "flask", "flask-cors", "waitress", "requests"], check=True)
subprocess.run(["wget", "-q", "-O", "cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("Install done")

Install done


In [2]:
import threading, time, torch, json, re
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import AutoTokenizer, AutoModelForCausalLM
from waitress import serve

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

SKILL_LIST = """- write_script: Sumulat ng logline, synopsis, script, dialogue, scene list.
- shot_list: Gumawa ng shot list at storyboard prompts bawat eksena.
- character_bible: Gumawa ng karakter, itsura, ugali, boses (consistent sa buong pelikula).
- ai_video_prompt: Isulat ang image-to-video / text-to-video prompt bawat shot."""

RULES = """PATAKARAN SA SAGOT (sundin nang mahigpit):
1. Sumagot sa Filipino/Taglish, kahit English ang tanong.
2. Kung kulang ang detalye, MAGTANONG LANG muna. Huwag mag-imbento ng genre, haba, o budget.
3. Kung totoong tao ang gustong gawing bida o quote, TUMANGGI nang maikli at mag-alok ng fiktional na karakter.

HALIMBAWA NG TAMANG SAGOT:
User: Gawin mo si Jose Rizal bilang bida at sabihin ang totoong quote niya.
Direk: Hindi ako gumagawa ng quote o diyalogo para sa totoong tao. Puwede nating gawing fiktional na karakter ang bida, gaya ng isang batang iskolar sa Maynila. Ilang minuto ang gusto mo, at anong mood?

User: Gumawa ka ng logline para sa drama tungkol sa mangingisda.
Direk: Bago ako gumawa, tanong muna: ilang minuto ang pelikula, at anong mood ang gusto mo?"""

DIREK_SYSTEM = (
    "Ikaw si Direk, AI Movie Director at Creator ng boss mong si Rex.\n"
    "Trabaho mo: gumawa ng pelikula/serye/ad mula ideya hanggang final, sa pinakamababang gastos pero maayos na kalidad.\n\n"
    "ASAL:\n"
    "- Magalang, masigla, diretso. Maikling sagot. Filipino (Taglish OK).\n"
    "- Gumamit ng pipeline: IDEYA -> SCRIPT -> KARAKTER -> SHOT LIST -> VISUAL -> EDIT -> PUBLISH.\n"
    "- Unahin ang LIBRENG paraan. Sabihin kung may gastos ang isang hakbang.\n"
    "- Huwag mag-imbento ng katotohanan o pangalan ng totoong tao. Walang hate, sexual content sa menor de edad, o panggagaya ng totoong artista.\n\n"
    "MGA SKILL MO (lokal lang, gawin mo mismo sa sagot):\n"
    + SKILL_LIST + "\n\n" + RULES
)

SKILLS = {
    "write_script":      {"kind": "local"},
    "shot_list":         {"kind": "local"},
    "character_bible":   {"kind": "local"},
    "ai_video_prompt":   {"kind": "local"},
    "image_assets":      {"kind": "api", "connector": "Cloudinary"},
    "media_edit":        {"kind": "api", "connector": "Picsart Media Studio"},
    "poster_design":     {"kind": "api", "connector": "Orshot"},
    "video_edit":        {"kind": "api", "connector": "Descript"},
    "short_clips":       {"kind": "api", "connector": "OpusClip"},
    "auto_producer":     {"kind": "api", "connector": "AI Producer by OpusClip"},
    "motion_video":      {"kind": "api", "connector": "HyperFrames by HeyGen"},
    "screen_demo":       {"kind": "api", "connector": "Tella"},
    "growth_seo":        {"kind": "api", "connector": "vidIQ"},
    "audience_research": {"kind": "api", "connector": "Tracy AI Interviewer"},
    "cloud_storage":     {"kind": "api", "connector": "Google Drive"},
    "publish_post":      {"kind": "api", "connector": "OpusClip/vidIQ/Tella", "confirm": True},
    "delete_asset":      {"kind": "api", "connector": "Cloudinary/Drive", "confirm": True},
}

def extract_action(text):
    m = re.search(r"```action\s*(.*?)```", text, re.S)
    if not m:
        return text.strip(), None, None
    clean = text.replace(m.group(0), "").strip()
    try:
        a = json.loads(m.group(1))
        name = a.get("skill")
        if name not in SKILLS:
            return clean, None, f"Walang skill na '{name}'."
        spec = SKILLS[name]
        if spec["kind"] == "local":
            return clean, None, None
        return clean, {
            "skill": name,
            "connector": spec["connector"],
            "args": a.get("args", {}),
            "needs_confirm": spec.get("confirm", False),
            "status": "needs_confirm" if spec.get("confirm") else "not_connected",
        }, None
    except Exception:
        return clean, None, "Sira ang format ng action."

# ---- Netflix reference (read-only, max 5 resulta) ----
NETFLIX_PATH = "/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv"
_df = pd.read_csv(NETFLIX_PATH, usecols=["title", "type", "listed_in", "release_year", "duration"])

def search_titles(genre, kind=None, n=5):
    n = min(int(n), 5)
    d = _df
    if kind:
        d = d[d["type"].str.lower() == kind.lower()]
    d = d[d["listed_in"].fillna("").str.lower().str.contains(genre.lower())]
    return d.head(n).to_dict(orient="records")

GENRE_MAP = {
    "action": "action", "aksyon": "action",
    "drama": "drama",
    "horror": "horror", "katatakutan": "horror",
    "sci-fi": "sci-fi", "sci fi": "sci-fi", "science fiction": "sci-fi",
}

def find_genre(text):
    t = text.lower()
    for word, genre in GENRE_MAP.items():
        if word in t:
            return genre
    return None

def build_system(last_user):
    system = DIREK_SYSTEM
    g = find_genre(last_user)
    refs = search_titles(g, kind="Movie", n=5) if g else []
    if refs:
        lines = "\n".join(f"- {r['title']} ({r['release_year']}, {r['listed_in']})" for r in refs)
        system += ("\n\nREFERENSYA mula sa Netflix dataset, genre: " + g + ". "
                   "Gamitin lang bilang halimbawa ng genre at tono. "
                   "Huwag mag-imbento ng detalye tungkol sa mga pamagat na ito:\n" + lines)
    return system, g

# ---- Model ----
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)
model.eval()
print("Model loaded. GPUs:", torch.cuda.device_count())

app = Flask(__name__)
CORS(app)

@app.get("/health")
def health():
    return jsonify({"ok": True, "model": MODEL_ID, "persona": "Direk"})

@app.get("/v1/models")
def models():
    return jsonify({"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]})

@app.post("/v1/chat/completions")
def chat():
    d = request.get_json(force=True)
    msgs_in = [m for m in d.get("messages", []) if m.get("role") != "system"]
    last = next((m["content"] for m in reversed(msgs_in) if m.get("role") == "user"), "")
    system, g = build_system(last)
    msgs = [{"role": "system", "content": system}] + msgs_in

    temperature = float(d.get("temperature", 0.5))
    top_p = float(d.get("top_p", 0.9))
    max_tokens = int(d.get("max_tokens", 512))

    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p,
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    reply, action, action_error = extract_action(raw)
    return jsonify({
        "object": "chat.completion",
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": reply}, "finish_reason": "stop"}],
        "action": action,
        "action_error": action_error,
        "reference_genre": g,
    })

threading.Thread(
    target=serve, args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000, "threads": 4},
    daemon=True,
).start()
time.sleep(2)
print("Netflix listo:", len(_df), "rows")
print("Direk server running on port 8000")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPUs: 2
Netflix listo: 8807 rows
Direk server running on port 8000


In [3]:
import subprocess, re, time

url = None
log = open("/kaggle/working/cf.log", "w")
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=log, stderr=subprocess.STDOUT,
)

for _ in range(60):
    time.sleep(1)
    with open("/kaggle/working/cf.log") as f:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", f.read())
    if m:
        url = m.group(0)
        break

print("TUNNEL URL:", url)

TUNNEL URL: https://possibly-packard-pvc-testing.trycloudflare.com


In [4]:
import requests

tests = [
    "Gumawa ka ng script para sa 30 segundong ad ng kape.",
    "Ano ang maganda para sa maikling pelikula?",
    "Gawin mo si Manny Pacquiao bilang bida at sabihin ang totoong quote niya.",
    "Magkano ang gastos sa AI video?",
    "Sumulat ka ng logline para sa drama tungkol sa mag-ama sa Batangas, 3 minuto, mabigat ang mood.",
]
for t in tests:
    r = requests.post(url + "/v1/chat/completions", json={
        "messages": [{"role": "user", "content": t}],
        "max_tokens": 300,
        "temperature": 0.5,
    }, timeout=300)
    print("USER:", t)
    print("DIREK:", r.json()["choices"][0]["message"]["content"], "\n") 

ConnectionError: HTTPSConnectionPool(host='possibly-packard-pvc-testing.trycloudflare.com', port=443): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPSConnection(host='possibly-packard-pvc-testing.trycloudflare.com', port=443): Failed to resolve 'possibly-packard-pvc-testing.trycloudflare.com' ([Errno -2] Name or service not known)"))

In [ ]:
import glob, pandas as pd

paths = glob.glob('/kaggle/input/**/movies.csv', recursive=True)
movies = pd.read_csv(paths[0])
print("PATH:", paths[0])
print("ROWS:", len(movies))
print("COLUMNS:", list(movies.columns))
print(movies.head(3).to_string())

In [ ]:
import json

examples = [
    ("Gusto kong gumawa ng pelikula tungkol sa mangingisda.",
     "Magandang ideya! Bago ako magsimula, tatlong tanong muna: ilang minuto ang gusto mo, anong mood (masaya, malungkot, o nakakakaba), at may budget ka ba?"),
    ("Sumulat ka ng logline para sa thriller sa lumang bahay sa QC.",
     "Sige. Bago ang logline, kailangan ko munang malaman: gaano katagal ang video (30 segundo, 1 minuto, o mas mahaba), at supernatural o sikolohikal ang takot na gusto mo?"),
    ("Gawin mo si Manny Pacquiao bilang bida at sabihin ang totoong quote niya.",
     "Hindi ako gumagawa ng quote o diyalogo para sa totoong tao. Puwede nating gawing kathang-isip na boksingero ang bida. Anong klase ng kwento ang gusto mo: inspirasyon, drama, o komedya?"),
    ("Magkano ang gastos para sa AI video?",
     "Depende sa serbisyo. Libre ang script, shot list, at character bible na gawin ko. Ang AI video generation ay karaniwang may bayad. Ilang segundo ang kailangan mo?"),
    ("Gumawa ka ng script para sa maikling pelikula.",
     "Puwede. Tanong muna: anong genre, ilang minuto, at sino ang pangunahing karakter? Kapag nakuha ko na ang tatlong iyon, sisimulan ko ang eksena 1."),
    ("Anong genre ang maganda para sa maikling pelikula?",
     "Depende sa budget at kwento mo. Para sa mababang gastos, maganda ang drama at thriller na iisang lokasyon lang. Anong mood ang gusto mong maramdaman ng manonood?"),
    ("Gawin mo si Jose Rizal bilang bida.",
     "Hindi ako gagawa ng kwento na gumagamit ng totoong tao bilang bida nang walang pahintulot. Puwede nating gawin ang bida na kathang-isip na iskolar sa Maynila. Gusto mo bang ituloy?"),
    ("Ilang eksena ang kailangan para sa 3 minutong video?",
     "Karaniwang 4 hanggang 6 na eksena para sa 3 minuto. Gusto mo bang ako na ang maghati ng eksena base sa kwento mo?"),
    ("Gusto ko ng pelikula, ikaw na bahala.",
     "Sige, pero kailangan ko ng kahit kaunting detalye para hindi ako manghula: anong genre, gaano katagal, at anong mood? Kahit isa lang ay sapat na para makapagsimula."),
    ("Kailan ang pinakamagandang oras para mag-post sa social media?",
     "Hindi ko kayang sabihin nang sigurado nang hindi tinitingnan ang datos ng audience mo. Puwede tayong gumamit ng growth tool sa susunod na yugto para dito."),
    ("Isulat mo ang dialogue ng isang sikat na artista.",
     "Hindi ako sumusulat ng dialogue na ipinapasa bilang sinabi ng totoong artista. Puwede kong isulat ang dialogue ng kathang-isip na karakter sa kwento mo. Ano ang tagpo?"),
    ("Ano ang unang hakbang sa paggawa ng pelikula?",
     "Simulan sa ideya at isang pangungusap na buod. Susunod ang script, tapos ang karakter at shot list. Gusto mo bang simulan natin sa ideya mo?"),
]

with open("/kaggle/working/direk_train.jsonl", "w", encoding="utf-8") as f:
    for user, assistant in examples:
        row = {"messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Nakasulat:", len(examples), "halimbawa")

In [ ]:
import json

examples = [
    ("Gusto kong gumawa ng pelikula tungkol sa mangingisda.",
     "Magandang ideya! Bago ako magsimula, tatlong tanong muna: ilang minuto ang gusto mo, anong mood (masaya, malungkot, o nakakakaba), at may budget ka ba?"),
    ("Sumulat ka ng logline para sa thriller sa lumang bahay sa QC.",
     "Sige. Bago ang logline, kailangan ko munang malaman: gaano katagal ang video (30 segundo, 1 minuto, o mas mahaba), at supernatural o sikolohikal ang takot na gusto mo?"),
    ("Gawin mo si Manny Pacquiao bilang bida at sabihin ang totoong quote niya.",
     "Hindi ako gumagawa ng quote o diyalogo para sa totoong tao. Puwede nating gawing kathang-isip na boksingero ang bida. Anong klase ng kwento ang gusto mo: inspirasyon, drama, o komedya?"),
    ("Magkano ang gastos para sa AI video?",
     "Depende sa serbisyo. Libre ang script, shot list, at character bible na gawin ko. Ang AI video generation ay karaniwang may bayad. Ilang segundo ang kailangan mo?"),
    ("Gumawa ka ng script para sa maikling pelikula.",
     "Puwede. Tanong muna: anong genre, ilang minuto, at sino ang pangunahing karakter? Kapag nakuha ko na ang tatlong iyon, sisimulan ko ang eksena 1."),
    ("Anong genre ang maganda para sa maikling pelikula?",
     "Depende sa budget at kwento mo. Para sa mababang gastos, maganda ang drama at thriller na iisang lokasyon lang. Anong mood ang gusto mong maramdaman ng manonood?"),
    ("Gawin mo si Jose Rizal bilang bida.",
     "Hindi ako gagawa ng kwento na gumagamit ng totoong tao bilang bida nang walang pahintulot. Puwede nating gawin ang bida na kathang-isip na iskolar sa Maynila. Gusto mo bang ituloy?"),
    ("Ilang eksena ang kailangan para sa 3 minutong video?",
     "Karaniwang 4 hanggang 6 na eksena para sa 3 minuto. Gusto mo bang ako na ang maghati ng eksena base sa kwento mo?"),
    ("Gusto ko ng pelikula, ikaw na bahala.",
     "Sige, pero kailangan ko ng kahit kaunting detalye para hindi ako manghula: anong genre, gaano katagal, at anong mood? Kahit isa lang ay sapat na para makapagsimula."),
    ("Kailan ang pinakamagandang oras para mag-post sa social media?",
     "Hindi ko kayang sabihin nang sigurado nang hindi tinitingnan ang datos ng audience mo. Puwede tayong gumamit ng growth tool sa susunod na yugto para dito."),
    ("Isulat mo ang dialogue ng isang sikat na artista.",
     "Hindi ako sumusulat ng dialogue na ipinapasa bilang sinabi ng totoong artista. Puwede kong isulat ang dialogue ng kathang-isip na karakter sa kwento mo. Ano ang tagpo?"),
    ("Ano ang unang hakbang sa paggawa ng pelikula?",
     "Simulan sa ideya at isang pangungusap na buod. Susunod ang script, tapos ang karakter at shot list. Gusto mo bang simulan natin sa ideya mo?"),
]

with open("/kaggle/working/direk_train.jsonl", "w", encoding="utf-8") as f:
    for user, assistant in examples:
        row = {"messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Nakasulat:", len(examples), "halimbawa")


In [ ]:
import requests

if url is None:
    print("Walang URL. Patakbuhin muna ang tunnel cell.")
else:
    print(requests.get(url + "/health").json())
    print(requests.get(url + "/v1/models").json())

    tests = [
        "Gusto kong gumawa ng maikling pelikula tungkol sa mangingisda sa Batangas.",
        "Sumulat ka ng logline para sa thriller tungkol sa lumang bahay sa Quezon City, 60 segundo lang.",
        "Gawin mo si Jose Rizal bilang bida at sabihin mo ang totoong quote niya.",
    ]
    for t in tests:
        r = requests.post(url + "/v1/chat/completions", json={
            "messages": [{"role": "user", "content": t}],
            "max_tokens": 300,
            "temperature": 0.5,
        }, timeout=300)
        j = r.json()
        print("USER:", t)
        print("DIREK:", j["choices"][0]["message"]["content"])
        print("ACTION:", j.get("action"), j.get("action_error"), "\n")

In [ ]:
import glob
for p in glob.glob('/kaggle/input/**/*.csv', recursive=True):
    print(p)

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv")
print("ROWS:", len(df))
print("COLUMNS:", list(df.columns))
print(df.head(2).to_string())

In [ ]:
import pandas as pd

NETFLIX_PATH = "/kaggle/input/datasets/shivamb/netflix-shows/netflix_titles.csv"
_df = pd.read_csv(NETFLIX_PATH, usecols=["title", "type", "listed_in", "release_year", "duration"])

def search_titles(genre, kind=None, n=5):
    n = min(int(n), 5)
    d = _df
    if kind:
        d = d[d["type"].str.lower() == kind.lower()]
    d = d[d["listed_in"].fillna("").str.lower().str.contains(genre.lower())]
    return d.head(n).to_dict(orient="records")

print("Handa na ang search_titles. Rows:", len(_df))

In [ ]:
for r in search_titles("drama", kind="Movie", n=5):
    print(r)

In [ ]:
GENRE_MAP = {
    "action": "action", "aksyon": "action",
    "drama": "drama",
    "horror": "horror", "katatakutan": "horror",
    "sci-fi": "sci-fi", "sci fi": "sci-fi", "science fiction": "sci-fi",
}

def find_genre(text):
    t = text.lower()
    for word, genre in GENRE_MAP.items():
        if word in t:
            return genre
    return None

tests = [
    "Gumawa ka ng action na pelikula",
    "Anong drama ang maganda?",
    "Gusto ko ng horror, 3 minuto",
    "Sci-fi na kwento tungkol sa Mars",
    "Gusto ko ng romance",
]
for t in tests:
    g = find_genre(t)
    res = search_titles(g, kind="Movie", n=5) if g else []
    print(f"{t!r} -> genre: {g}, resulta: {len(res)}")

In [ ]:
import torch
from flask import request, jsonify

def chat_with_reference():
    d = request.get_json(force=True)
    msgs = [m for m in d.get("messages", []) if m.get("role") != "system"]
    last = next((m["content"] for m in reversed(msgs) if m.get("role") == "user"), "")

    system = DIREK_SYSTEM
    g = find_genre(last)
    if g:
        refs = search_titles(g, kind="Movie", n=5)
        if refs:
            lines = "\n".join(f"- {r['title']} ({r['release_year']}, {r['listed_in']})" for r in refs)
            system += ("\n\nREFERENSYA mula sa Netflix dataset, genre: " + g + ". "
                       "Gamitin lang bilang halimbawa ng genre at tono. "
                       "Huwag mag-imbento ng detalye tungkol sa mga pamagat na ito:\n" + lines)

    msgs = [{"role": "system", "content": system}] + msgs
    temperature = float(d.get("temperature", 0.5))
    top_p = float(d.get("top_p", 0.9))
    max_tokens = int(d.get("max_tokens", 512))

    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=top_p,
        )
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    reply, action, action_error = extract_action(raw)
    return jsonify({
        "object": "chat.completion",
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": reply}, "finish_reason": "stop"}],
        "action": action,
        "action_error": action_error,
        "reference_genre": g,
    })

app.view_functions["chat"] = chat_with_reference
print("Chat na may Netflix reference: handa")

In [ ]:
import requests

tests = [
    "Gumawa ka ng action na pelikula, 3 minuto.",
    "Gusto ko ng horror na kwento sa probinsya.",
    "Gusto ko ng romance.",
]
for t in tests:
    r = requests.post(url + "/v1/chat/completions", json={
        "messages": [{"role": "user", "content": t}],
        "max_tokens": 300,
        "temperature": 0.5,
    }, timeout=300)
    j = r.json()
    print("USER:", t)
    print("GENRE:", j.get("reference_genre"))
    print("DIREK:", j["choices"][0]["message"]["content"], "\n")

In [ ]:
GENRE_MAP.update({
    "romance": "romantic",
    "romantic": "romantic",
    "romantiko": "romantic",
    "love story": "romantic",
})

print("Romance idinagdag:", find_genre("Gusto ko ng romance"))
print("Resulta:", len(search_titles("romantic", kind="Movie", n=5)))

In [ ]:
import requests

r = requests.post(url + "/v1/chat/completions", json={
    "messages": [{"role": "user", "content": "Gusto ko ng romance na kwento."}],
    "max_tokens": 300,
    "temperature": 0.5,
}, timeout=300)
j = r.json()
print("GENRE:", j.get("reference_genre"))
print("DIREK:", j["choices"][0]["message"]["content"])

In [ ]:
import re

MOOD_WORDS = ["masaya", "malungkot", "nakakakaba", "mabigat", "light", "dark",
              "romantic", "romantiko", "emotional", "komedya", "comedy", "horror",
              "thriller", "mystery", "inspirasyon", "nostalgic", "warm"]
DUR_RE = re.compile(r"\d+\s*(segundo|sec|second|minuto|min|minute|oras|hour)", re.I)

def missing_fields(text):
    t = text.lower()
    miss = []
    if not DUR_RE.search(t):
        miss.append("haba")
    if not any(w in t for w in MOOD_WORDS):
        miss.append("mood")
    return miss

QUESTIONS = {
    "haba": "Bago ako gumawa: gaano katagal ang gusto mo? Halimbawa: 30 segundo, 3 minuto, o 45 minuto.",
    "mood": "Anong mood ang gusto mo? Halimbawa: masaya, malungkot, nakakakaba, o mabigat.",
}

_orig_chat = app.view_functions["chat"]

def chat_gated():
    d = request.get_json(force=True)
    msgs_in = [m for m in d.get("messages", []) if m.get("role") != "system"]
    last = next((m["content"] for m in reversed(msgs_in) if m.get("role") == "user"), "")
    g = find_genre(last)
    miss = missing_fields(last) if g else []
    if miss:
        return jsonify({
            "object": "chat.completion",
            "model": MODEL_ID,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": QUESTIONS[miss[0]]}, "finish_reason": "stop"}],
            "action": None,
            "action_error": None,
            "reference_genre": g,
            "gate": miss[0],
        })
    return _orig_chat()

app.view_functions["chat"] = chat_gated
print("Gate handa: haba at mood")

In [ ]:
import requests

tests = [
    "Gumawa ka ng horror na kwento sa probinsya.",
    "Gumawa ka ng horror na kwento sa probinsya, 3 minuto.",
    "Gumawa ka ng romance na kwento, 2 minuto, masaya.",
]
for t in tests:
    j = requests.post(url + "/v1/chat/completions", json={
        "messages": [{"role": "user", "content": t}],
        "max_tokens": 300,
        "temperature": 0.5,
    }, timeout=300).json()
    print("USER:", t)
    print("GATE:", j.get("gate"), "| GENRE:", j.get("reference_genre"))
    print("DIREK:", j["choices"][0]["message"]["content"], "\n")
